In [ ]:
import json
import pandas as pd

def flatten_json(y, parent_key=""):
    """
    Flatten a nested JSON object into a flat dict whose keys are the CSV header notations.
    For top-level keys, if the key is not "SchoolYear", prefix with "/ed-fi/schools/".
    Nested keys are joined with a period and array indices are added in square brackets.
    """
    items = {}
    if isinstance(y, dict):
        for k, v in y.items():
            if parent_key == "":
                # top-level: keep "SchoolYear" as is; for others add prefix.
                new_key = k if k == "SchoolYear" else f"/ed-fi/schools/{k}"
            else:
                new_key = f"{parent_key}.{k}"
            if isinstance(v, dict):
                items.update(flatten_json(v, new_key))
            elif isinstance(v, list):
                for i, elem in enumerate(v):
                    list_key = f"{new_key}[{i}]"
                    if isinstance(elem, (dict, list)):
                        items.update(flatten_json(elem, list_key))
                    else:
                        items[list_key] = elem
            else:
                items[new_key] = v
    elif isinstance(y, list):
        for i, elem in enumerate(y):
            new_key = f"{parent_key}[{i}]"
            if isinstance(elem, (dict, list)):
                items.update(flatten_json(elem, new_key))
            else:
                items[new_key] = elem
    else:
        items[parent_key] = y
    return items

def json_array_to_csv(json_file_path, csv_file_path):
    # Load JSON array from file
    with open(json_file_path, 'r') as f:
        data = json.load(f)
    
    # Flatten each JSON record
    flat_records = [flatten_json(record) for record in data]
    
    # Create a DataFrame; union of all keys will be used as columns
    df = pd.DataFrame(flat_records)
    
    # Save to CSV
    df.to_csv(csv_file_path, index=False)
    print(f"CSV file saved to {csv_file_path}")



In [ ]:
import os
import requests


client_id = os.getenv('EDFI_API_CLIENT_ID')
client_secret = os.getenv('EDFI_API_CLIENT_SECRET')
base_url = 'https://api.ed-fi.org:443/v7.2/api/data/v3'

def get_access_token(client_id, client_secret):
    url = "https://api.ed-fi.org/v7.2/api/oauth/token"
    data = {
        'grant_type': 'client_credentials',
        'client_id': client_id,
        'client_secret': client_secret
    }
    response = requests.post(url, data=data)
    response.raise_for_status()
    return response.json().get('access_token')


if not client_id or not client_secret :
    raise EnvironmentError("Missing one or more environment variables: EDFI_API_CLIENT_ID, EDFI_API_CLIENT_SECRET")

endpoint = "/ed-fi/calendars?offset=0&limit=20"
url = f"{base_url}{endpoint}"
token = get_access_token(client_id, client_secret)
headers = {
    'Authorization': f'Bearer {token}',
    'Accept': 'application/json',
    'Content-Type': 'application/json',

}
response = requests.get(url, headers=headers)

if response.ok:
    print(response.json())
else:
    response.raise_for_status()

In [ ]:
# Convert the JSON response to a CSV file
json_data = response.json()
flat_records = [flatten_json(record, parent_key='/ed-fi/calendars') for record in json_data]
df = pd.DataFrame(flat_records)
csv_file_path = './data/calendars.csv'
df.to_csv(csv_file_path, index=False)
print(f"CSV file saved to {csv_file_path}")

In [ ]:
import json

input_file = './output/calendars/calendars.jsonl'
output_file = './output/calendars/calendars_updated.jsonl'

with open(input_file, 'r') as fin, open(output_file, 'w') as fout:
    for line in fin:
        line = line.strip()
        if not line:
            continue
        try:
            record = json.loads(line)
            # Convert schoolId from string to int if present.
            if ("schoolReference" in record and 
                "schoolId" in record["schoolReference"]):
                try:
                    record["schoolReference"]["schoolId"] = int(record["schoolReference"]["schoolId"])
                except ValueError:
                    # Log or leave unchanged if conversion fails.
                    pass
            fout.write(json.dumps(record) + "\n")
        except Exception as e:
            print(f"Error processing line: {line}\n{e}")
            
print(f"Updated JSONL saved to {output_file}")